# 생각중인 요소
1. Tableau용 파생 컬럼 추가
    - member_year
    - member_month
    - member_quarter
    - age_group
    - income_group

2. value 컬럼 제거

3. profile 결측 고객 2,175명 제거 유지 여부

# 데이터 전처리

In [113]:
import numpy as np
import pandas as pd
from IPython.display import display
import warnings
import platform
import matplotlib.pyplot as plt

# 출력 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams[
        'font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


## 사용 함수

In [114]:
# 컬럼 정보 간단 표현
def check_basic_info(df, df_name, exclude_cols=None):
    
    print(f"\n{'='*80}")
    print(f"{df_name}의 컬럼 정보 / 결측치 확인 정보 요약")
    print(f"{'='*80}\n")


    # 제외할 컬럼 반영
    df_copied = df.copy()
    if exclude_cols:
        df_copied = df_copied.drop(columns=exclude_cols, errors='ignore')

    # dict, list, set 같은 해시 불가능 값이 들어있는 컬럼은 문자열로 변환
    for col in df_copied.columns:
        try:
            df_copied[col].nunique(dropna=True)
        except TypeError:
            df_copied[col] = df_copied[col].astype(str)
    
    # 1. 전체 요약
    overview_df = pd.DataFrame({
        '항목': ['행 개수', '열 개수', '중복 행 개수'],
        '값': [df_copied.shape[0], df_copied.shape[1], df_copied.duplicated().sum()]
    })
    
    # 2. 컬럼별 요약
    summary_df = pd.DataFrame({
        '데이터타입': df_copied.dtypes.astype(str),
        '행 개수': df_copied.count(),
        '행 비율(%)': (df_copied.count() / len(df) * 100).round(2),
        '결측치 개수': df_copied.isnull().sum(),
        '결측치 비율(%)': (df_copied.isnull().sum() / len(df) * 100).round(2),
        '고유값 개수': df_copied.nunique(dropna=True)
    })
    
    # 3. 보기 좋게 정렬
    summary_df = summary_df.sort_values(
        by=['결측치 개수', '고유값 개수'],
        ascending=[False, False]
    )
    
    print("[전체 요약]")
    display(overview_df)
    
    print("[컬럼별 요약]")
    display(summary_df)

    print("[테이블 요약]")
    display(df.head())

In [115]:
# 중복값 확인
def check_id_duplicates(df, col_name, df_name):
    print(f"\n{'='*80}")
    print(f"{df_name}의 값 중복 확인")
    print(f"{'='*80}")
    
    print("전체 행 수:", len(df))
    print(f"{col_name} 고유 개수:", df[col_name].nunique())
    print(f"중복 {col_name} 개수:", df[col_name].duplicated().sum())

In [116]:
# 컬럼 분포 확인
def check_category_summary(df, df_name, col_name):
    
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 범주 확인")
    print(f"{'='*80}")
    
    summary_df = df[col_name].value_counts(dropna=False).reset_index()
    summary_df.columns = [col_name, '개수']
    summary_df['비율(%)'] = (summary_df['개수'] / len(df) * 100).round(2)
    
    display(summary_df.head(10))

# 테이블 전처리

In [117]:
# ============================================================
# 1. 원본 데이터 로드
# ============================================================
df_portfolio = pd.read_csv("data/portfolio.csv", index_col=0)
df_profile = pd.read_csv("data/profile.csv", index_col=0)
df_transcript = pd.read_csv("data/transcript.csv", index_col=0)

## df_portfolio 전처리

In [118]:
# ============================================================
# 확인 및 복제
# ============================================================
print("[portfolio]")
df_portfolio_copy = df_portfolio.copy()
check_basic_info(df_portfolio_copy, "portfolio")

[portfolio]

portfolio의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,10
1,열 개수,6
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
id,str,10,100.0,0,0.0,10
reward,int64,10,100.0,0,0.0,5
difficulty,int64,10,100.0,0,0.0,5
duration,int64,10,100.0,0,0.0,5
channels,str,10,100.0,0,0.0,4
offer_type,str,10,100.0,0,0.0,3


[테이블 요약]


,reward,channels,difficulty,duration,offer_type,id
0,10,"['email', 'mobile', 'social']",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"['web', 'email', 'mobile', 'social']",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"['web', 'email', 'mobile']",0,4,informational,3f207df678b143eea3cee63160fa8bed
3,5,"['web', 'email', 'mobile']",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,5,"['web', 'email']",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7


In [119]:
# ============================================================
# 컬럼명 정리
# ============================================================
df_portfolio_clean = (
    df_portfolio_copy
    .rename(columns={
    'id': 'offer_id',
    'reward': 'offer_reward'
    })
)
display(df_portfolio_clean.head())

,offer_reward,channels,difficulty,duration,offer_type,offer_id
0,10,"['email', 'mobile', 'social']",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"['web', 'email', 'mobile', 'social']",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"['web', 'email', 'mobile']",0,4,informational,3f207df678b143eea3cee63160fa8bed
3,5,"['web', 'email', 'mobile']",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,5,"['web', 'email']",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7


In [120]:
# ============================================================
# channels 문자열을 분리
# ============================================================
channel_list = ['web', 'email', 'mobile', 'social']


for ch in channel_list:
    df_portfolio_clean[ch] = (
        df_portfolio_clean['channels']
        .str
        .contains(ch)
        .astype(int)
    )

display(df_portfolio_clean.head())

,offer_reward,channels,difficulty,duration,offer_type,offer_id,web,email,mobile,social
0,10,"['email', 'mobile', 'social']",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd,0,1,1,1
1,10,"['web', 'email', 'mobile', 'social']",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0,1,1,1,1
2,0,"['web', 'email', 'mobile']",0,4,informational,3f207df678b143eea3cee63160fa8bed,1,1,1,0
3,5,"['web', 'email', 'mobile']",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9,1,1,1,0
4,5,"['web', 'email']",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7,1,1,0,0


## df_portfolio 전처리 최종 확인

In [121]:
check_basic_info(df_portfolio_clean, "portfolio_clean")
check_id_duplicates(df_portfolio_clean, 'offer_id', 'portfolio_clean')


portfolio_clean의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,10
1,열 개수,10
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
offer_id,str,10,100.0,0,0.0,10
offer_reward,int64,10,100.0,0,0.0,5
difficulty,int64,10,100.0,0,0.0,5
duration,int64,10,100.0,0,0.0,5
channels,str,10,100.0,0,0.0,4
offer_type,str,10,100.0,0,0.0,3
web,int64,10,100.0,0,0.0,2
mobile,int64,10,100.0,0,0.0,2
social,int64,10,100.0,0,0.0,2
email,int64,10,100.0,0,0.0,1


[테이블 요약]


,offer_reward,channels,difficulty,duration,offer_type,offer_id,web,email,mobile,social
0,10,"['email', 'mobile', 'social']",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd,0,1,1,1
1,10,"['web', 'email', 'mobile', 'social']",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0,1,1,1,1
2,0,"['web', 'email', 'mobile']",0,4,informational,3f207df678b143eea3cee63160fa8bed,1,1,1,0
3,5,"['web', 'email', 'mobile']",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9,1,1,1,0
4,5,"['web', 'email']",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7,1,1,0,0



portfolio_clean의 값 중복 확인
전체 행 수: 10
offer_id 고유 개수: 10
중복 offer_id 개수: 0


## df_profile 전처리

In [122]:
# ============================================================
# 확인 및 복제
# ============================================================
print("\n[profile]")
df_profile_copy = df_profile.copy()
check_basic_info(df_profile_copy, "profile")


[profile]

profile의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,17000
1,열 개수,5
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
income,float64,14825,87.21,2175,12.79,91
gender,str,14825,87.21,2175,12.79,3
id,str,17000,100.00,0,0.00,17000
became_member_on,int64,17000,100.00,0,0.00,1716
age,int64,17000,100.00,0,0.00,85


[테이블 요약]


,gender,age,id,became_member_on,income
0,NaN,118,68be06ca386d4c31939f3a4f0e3dd783,20170212,NaN
1,F,55,0610b486422d4921ae7d2bf64640c50b,20170715,112000.0
2,NaN,118,38fe809add3b4fcf9315a9694bb96ff5,20180712,NaN
3,F,75,78afa995795e4d85b5d9ceeca43f5fef,20170509,100000.0
4,NaN,118,a03223e636434f42ac4c3df47e8bac43,20170804,NaN


In [123]:
# ============================================================
# 컬럼명 정리
# ============================================================
df_profile_copy = (
    df_profile_copy
    .rename(columns={
    'id': 'customer_id'
}))

In [124]:
# ============================================================
# 가입일 날짜형 변환
# ============================================================
df_profile_copy['became_member_on'] = pd.to_datetime(
    df_profile_copy['became_member_on']
    .astype(str),
    format='%Y%m%d'
)

In [125]:
# ============================================================
# 가입일 파생 컬럼
# ============================================================
# 파생컬럼 필요시 사용
# # 가입 년도
# df_profile_copy['member_year'] = (
#     df_profile_copy['became_member_on']
#     .dt
#     .year
# )

# # 가입 월
# df_profile_copy['member_month'] = (
#     df_profile_copy['became_member_on'].
#     dt
#     .month
# )

# # 가입 분기
# df_profile_clean['member_quarter'] = (
#     df_profile_clean['became_member_on']
#     .dt
#     .quarter
# )

In [126]:
# ============================================================
# age 이상치 처리
# ============================================================
# age=118은 비정상값으로 보고 제거
# gender, income 결측도 함께 제거 대상에 포함


df_profile_copy['age'] = (
    df_profile_copy['age'].
    replace(118, np.nan)
)

# 제거 전 행 수 저장
before_len = len(df_profile_copy)

# age, gender, income 중 하나라도 결측이면 제거
df_profile_copy['age'] = df_profile_copy['age'].replace(118, np.nan)
df_profile_copy = df_profile_copy.dropna(subset=['age', 'gender', 'income']).copy()

# 제거 후 행 수 저장
after_len = len(df_profile_copy)

print("제거 전 행 수:", before_len)
print("제거 후 행 수:", after_len)
print("제거된 행 수:", before_len - after_len)

제거 전 행 수: 17000
제거 후 행 수: 14825
제거된 행 수: 2175


In [127]:
# ============================================================
# 컬럼 순서 정리
# ============================================================
df_profile_clean = df_profile_copy[
    [
        'customer_id',
        'gender', 'age', 'income', 'became_member_on'
    ]
]

## df_profile 전처리 최종확인

In [128]:
check_basic_info(df_profile_clean, "profile_clean")
check_id_duplicates(df_profile_clean, 'customer_id', 'profile_clean')


profile_clean의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,14825
1,열 개수,5
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
customer_id,str,14825,100.0,0,0.0,14825
became_member_on,datetime64[us],14825,100.0,0,0.0,1707
income,float64,14825,100.0,0,0.0,91
age,float64,14825,100.0,0,0.0,84
gender,str,14825,100.0,0,0.0,3


[테이블 요약]


,customer_id,gender,age,income,became_member_on
1,0610b486422d4921ae7d2bf64640c50b,F,55.0,112000.0,2017-07-15
3,78afa995795e4d85b5d9ceeca43f5fef,F,75.0,100000.0,2017-05-09
5,e2127556f4f64592b11af22de27a7932,M,68.0,70000.0,2018-04-26
8,389bc3fa690240e798340f5a15918d5c,M,65.0,53000.0,2018-02-09
12,2eeac8d8feae4a8cad5a6af0499a211d,M,58.0,51000.0,2017-11-11



profile_clean의 값 중복 확인
전체 행 수: 14825
customer_id 고유 개수: 14825
중복 customer_id 개수: 0


## df_transcript 전처리

In [129]:
# ============================================================
# 확인 및 복제
# ============================================================
print("[df_transcript]")
df_transcript_copy = df_transcript.copy()
check_basic_info(df_transcript_copy, "transcript")
check_category_summary(df_transcript_copy, "transcript", "event")

[df_transcript]

transcript의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,306534
1,열 개수,4
2,중복 행 개수,397


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
person,str,306534,100.0,0,0.0,17000
value,str,306534,100.0,0,0.0,5121
time,int64,306534,100.0,0,0.0,120
event,str,306534,100.0,0,0.0,4


[테이블 요약]


,person,event,value,time
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0
4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0



transcript의 event 범주 확인


,event,개수,비율(%)
0,transaction,138953,45.33
1,offer received,76277,24.88
2,offer viewed,57725,18.83
3,offer completed,33579,10.95


In [130]:
# ============================================================
# 컬럼명 정리
# ============================================================
df_transcript_copy = (
    df_transcript_copy
    .rename(columns={
    'person': 'customer_id'
}))

In [131]:
# ============================================================
# value 컬럼 파싱
# ============================================================
import ast
# ast란?

# ============================================================
# value 컬럼 딕셔너리로 형변환
# value 컬럼은 문자열처럼 보이지만 사실 딕셔너리 형태
# 먼저 컬럼을 딕셔너리의 형태로 변환한다.
# ============================================================
df_transcript_copy['value'] = (
    df_transcript_copy['value']
    .apply(ast.literal_eval))

print(df_transcript_copy['value'].apply(type).value_counts())

value
<class 'dict'>    306534
Name: count, dtype: int64


In [132]:
# ============================================================
# value 컬럼 키 구조 확인
# ============================================================
event_keys = sorted({
    key
    for d in df_transcript_copy['value']
    for key in d.keys()
})

print("value 컬럼 전체 key 종류:")
print(event_keys)

value 컬럼 전체 key 종류:
['amount', 'offer id', 'offer_id', 'reward']


In [133]:
# ============================================================
# value 컬럼(dict) 분리 및 주요 파생 컬럼 생성
# event별로 value 안에 들어 있는 값을 별도 컬럼으로 분리
# 'offer id'와 'offer_id'는 하나의 offer_id 컬럼으로 통합
# ============================================================
value_df = pd.DataFrame(df_transcript_copy['value'].tolist())
df_transcript_copy = pd.concat([df_transcript_copy, value_df], axis=1)

# offer id / offer_id 컬럼명을 하나로 통합
df_transcript_copy['offer_id'] = df_transcript_copy['offer id'].fillna(df_transcript_copy['offer_id'])

df_transcript_copy = (
    df_transcript_copy
    .rename(columns={
    'reward': 'event_reward'
    })
)

# 주요 컬럼 확인
display(
    df_transcript_copy[
        [
            'customer_id', 
            'event', 
            'time', 
            'offer_id', 
            'amount', 
            'event_reward']]
        .head()
)

,customer_id,event,time,offer_id,amount,event_reward
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN


In [134]:
# ============================================================
# event별 컬럼 결측 개수 확인
# ============================================================
print("="*80)
print("이벤트별 결측 확인")
print("="*80)

print(df_transcript_copy[['event', 'offer_id', 'amount', 'event_reward']].isna().groupby(df_transcript_copy['event']).sum())

이벤트별 결측 확인
                 event  offer_id  amount  event_reward
event                                                 
offer completed      0         0   33579             0
offer received       0         0   76277         76277
offer viewed         0         0   57725         57725
transaction          0    138953       0        138953


In [135]:
# ============================================================
# 중복 컬럼 결측 개수 확인
# ============================================================

# 중복 확인용 value 비교 컬럼 생성
df_transcript_copy['value_str'] = df_transcript_copy['value'].astype(str)

# 중복 기준
dup_cols = ['customer_id', 'event', 'time', 'offer_id', 'event_reward', 'value_str']

# 중복 행 추출
dup_df = df_transcript_copy[
    df_transcript_copy.duplicated(subset=dup_cols, keep=False)
].sort_values(['customer_id', 'time', 'offer_id'])

print("중복에 포함된 전체 행 수:", len(dup_df))
print()

print("[중복 event 분포]")
print(dup_df['event'].value_counts())
print()

print("[중복 횟수 분포]")
print(
    df_transcript_copy
    .groupby(dup_cols)
    .size()
    .reset_index(name='dup_count')
    .query("dup_count > 1")['dup_count']
    .value_counts()
    .sort_index()
)

display(
    dup_df[
        ['customer_id', 'event', 'time', 'offer_id', 'event_reward', 'value']
    ].head(5)
)

중복에 포함된 전체 행 수: 793

[중복 event 분포]
event
offer completed    793
Name: count, dtype: int64

[중복 횟수 분포]
dup_count
2    395
3      1
Name: count, dtype: int64


,customer_id,event,time,offer_id,event_reward,value
218058,00d7c95f793a4212af44e632fdc1e431,offer completed,504,2906b810c7d4411798c6938adc9daaa5,2.0,{'offer_id': '2906b810c7d4411798c6938adc9daaa5...
218060,00d7c95f793a4212af44e632fdc1e431,offer completed,504,2906b810c7d4411798c6938adc9daaa5,2.0,{'offer_id': '2906b810c7d4411798c6938adc9daaa5...
220133,01925607d99c460996c281f17cdbb9e2,offer completed,510,4d5c57ea9a6940dd891ad53e9dbe8da0,10.0,{'offer_id': '4d5c57ea9a6940dd891ad53e9dbe8da0...
220134,01925607d99c460996c281f17cdbb9e2,offer completed,510,4d5c57ea9a6940dd891ad53e9dbe8da0,10.0,{'offer_id': '4d5c57ea9a6940dd891ad53e9dbe8da0...
171646,01956670cf414b309675aa73368b94a9,offer completed,420,2906b810c7d4411798c6938adc9daaa5,2.0,{'offer_id': '2906b810c7d4411798c6938adc9daaa5...


offer completed 이벤트에서만 중복 문제 발생

중복구조: 
- 395개 경우는 2번씩 중복: 395 * 2 = 790
- 1개 경우는 3번 중복: 1 * 3 = 3
- 790 + 3 = 793

이는 향후 영향을 줄수 있기 때문에 중복 제거

In [136]:
# ============================================================
# 중복 제거
# ============================================================
df_transcript_drop = (
    df_transcript_copy
    .drop_duplicates(subset=dup_cols)
    .drop(columns=['offer id', 'value_str'], errors='ignore')
    .copy()
)

print("중복 제거 전 행 수:", len(df_transcript_copy))
print("중복 제거 후 행 수:", len(df_transcript_drop))
print("제거된 행 수:", len(df_transcript_copy) - len(df_transcript_drop))

중복 제거 전 행 수: 306534
중복 제거 후 행 수: 306137
제거된 행 수: 397


실제로 제거되는 행 수는 397개\
중복 제거할 때는 각 그룹에서 1개만 남기고 나머지를 지우기 때문

In [137]:
# ============================================================
# value 컬럼 드랍
# value 컬럼을 분리 및 주요 파생 컬럼 생성했기 때문에 제거
# ============================================================
df_transcript_copy = (
    df_transcript_copy
    .drop(columns=['value'], errors='ignore'))
df_transcript_copy.columns

Index(['customer_id', 'event', 'time', 'offer id', 'amount', 'offer_id',
       'event_reward', 'value_str'],
      dtype='str')

In [138]:
df_transcript_clean = df_transcript_drop[
    [
        'customer_id',
        'event',
        'time',
        'offer_id',
        'amount',
        'event_reward'
    ]
]

## df_transcript 전처리 최종확인

In [139]:
check_basic_info(df_transcript_clean, "transcript_clean")
check_id_duplicates(df_transcript_clean, 'customer_id', 'transcript_clean(customer_id)')
check_id_duplicates(df_transcript_clean, 'offer_id', 'transcript_clean(offer_id)')


transcript_clean의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,306137
1,열 개수,6
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
event_reward,float64,33182,10.84,272955,89.16,4
amount,float64,138953,45.39,167184,54.61,5103
offer_id,str,167184,54.61,138953,45.39,10
customer_id,str,306137,100.00,0,0.00,17000
time,int64,306137,100.00,0,0.00,120
event,str,306137,100.00,0,0.00,4


[테이블 요약]


,customer_id,event,time,offer_id,amount,event_reward
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN



transcript_clean(customer_id)의 값 중복 확인
전체 행 수: 306137
customer_id 고유 개수: 17000
중복 customer_id 개수: 289137

transcript_clean(offer_id)의 값 중복 확인
전체 행 수: 306137
offer_id 고유 개수: 10
중복 offer_id 개수: 306126


# 코드 통합

## transcript과 profile 매칭 확인

In [140]:
# 조인키: customer_id 
print("transcript customer_id 고유 개수:", df_transcript_clean['customer_id'].nunique())
print("profile customer_id 고유 개수:", df_profile_clean['customer_id'].nunique())

unmatched_customers = set(df_transcript_clean['customer_id']) - set(df_profile_clean['customer_id'])
print("profile에 없는 transcript 고객 수:", len(unmatched_customers))

transcript customer_id 고유 개수: 17000
profile customer_id 고유 개수: 14825
profile에 없는 transcript 고객 수: 2175


## transcript과 portfolio 매칭 확인

In [141]:
# 조인키: offer_id 
transcript_offer_ids = set(df_transcript_clean['offer_id'].dropna())
portfolio_offer_ids = set(df_portfolio_clean['offer_id'])

print("transcript 내 offer_id 개수:", len(transcript_offer_ids))
print("portfolio 내 offer_id 개수:", len(portfolio_offer_ids))
print("portfolio에 없는 transcript offer_id 수:", len(transcript_offer_ids - portfolio_offer_ids))

transcript 내 offer_id 개수: 10
portfolio 내 offer_id 개수: 10
portfolio에 없는 transcript offer_id 수: 0


## transcript + profile

In [142]:
# ============================================================
# transcript + profile 컬럼 조인
# ============================================================
df_transcript_profile = df_transcript_clean.merge(
    df_profile_clean,
    on='customer_id',
    how='left'
)

df_transcript_profile = df_transcript_profile[
    [
        # 로그 정보
        'customer_id', 'event', 'time', 'offer_id',

        # 이벤트/거래 정보
        'amount', 'event_reward',

        # 고객 정보
        'gender', 'age', 'income', 'became_member_on',
    ]
]


In [143]:
print("transcript 원본 행 수:", len(df_transcript_clean))
print("transcript + profile 행 수:", len(df_transcript_profile))
print("행 수 차이:", len(df_transcript_profile) - len(df_transcript_clean))

check_basic_info(df_transcript_profile, "transcript + profile", exclude_cols = 'value')
check_id_duplicates(df_transcript_profile, 'customer_id', 'transcript + profile')

transcript 원본 행 수: 306137
transcript + profile 행 수: 306137
행 수 차이: 0

transcript + profile의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,306137
1,열 개수,10
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
event_reward,float64,33182,10.84,272955,89.16,4
amount,float64,138953,45.39,167184,54.61,5103
offer_id,str,167184,54.61,138953,45.39,10
became_member_on,datetime64[us],272388,88.98,33749,11.02,1707
income,float64,272388,88.98,33749,11.02,91
age,float64,272388,88.98,33749,11.02,84
gender,str,272388,88.98,33749,11.02,3
customer_id,str,306137,100.00,0,0.00,17000
time,int64,306137,100.00,0,0.00,120
event,str,306137,100.00,0,0.00,4


[테이블 요약]


,customer_id,event,time,offer_id,amount,event_reward,gender,age,income,became_member_on
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,F,75.0,100000.0,2017-05-09
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,NaN,NaN,NaN,NaT
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,M,68.0,70000.0,2018-04-26
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,NaN,NaN,NaN,NaT
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,NaN,NaN,NaN,NaT



transcript + profile의 값 중복 확인
전체 행 수: 306137
customer_id 고유 개수: 17000
중복 customer_id 개수: 289137


## transcript + portfolio

In [144]:
# ============================================================
# transcript + portfolio 컬럼 조인
# ============================================================
df_transcript_portfolio = df_transcript_clean.merge(
    df_portfolio_clean,
    on='offer_id',
    how='left'
)

df_transcript_portfolio = df_transcript_portfolio[
    [
        # 로그 정보
        'customer_id', 'event', 'time', 'offer_id',

        # 이벤트/거래 정보
        'amount', 'event_reward',

        # 오퍼 정보
        'offer_reward', 'offer_type', 'difficulty', 'duration', 'channels',

        # 채널 정보
        'web', 'email', 'mobile', 'social',
    ]
]


In [145]:
print("transcript 원본 행 수:", len(df_transcript_clean))
print("transcript + portfolio 행 수:", len(df_transcript_portfolio))
print("행 수 차이:", len(df_transcript_portfolio) - len(df_transcript_clean))

check_basic_info(df_transcript_portfolio, "transcript + portfolio")
check_id_duplicates(df_transcript_portfolio, 'offer_id', 'transcript + portfolio')

transcript 원본 행 수: 306137
transcript + portfolio 행 수: 306137
행 수 차이: 0

transcript + portfolio의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,306137
1,열 개수,15
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
event_reward,float64,33182,10.84,272955,89.16,4
amount,float64,138953,45.39,167184,54.61,5103
offer_id,str,167184,54.61,138953,45.39,10
offer_reward,float64,167184,54.61,138953,45.39,5
difficulty,float64,167184,54.61,138953,45.39,5
duration,float64,167184,54.61,138953,45.39,5
channels,str,167184,54.61,138953,45.39,4
offer_type,str,167184,54.61,138953,45.39,3
web,float64,167184,54.61,138953,45.39,2
mobile,float64,167184,54.61,138953,45.39,2


[테이블 요약]


,customer_id,event,time,offer_id,amount,event_reward,offer_reward,offer_type,difficulty,duration,channels,web,email,mobile,social
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,5.0,bogo,5.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,5.0,discount,20.0,10.0,"['web', 'email']",1.0,1.0,0.0,0.0
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,2.0,discount,10.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,2.0,discount,10.0,10.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,10.0,bogo,10.0,5.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0



transcript + portfolio의 값 중복 확인
전체 행 수: 306137
offer_id 고유 개수: 10
중복 offer_id 개수: 306126


## transcript + portfolio + profile

In [146]:
# ============================================================
# transcript + portfolio + profile 컬럼 조인
# ============================================================
df_transcript_portfolio_profile = (
    df_transcript_clean
    .merge(df_portfolio_clean, on='offer_id', how='left')
    .merge(df_profile_clean, on='customer_id', how='left')
)

df_transcript_portfolio_profile = df_transcript_portfolio_profile[
    [
        # 로그 정보
        'customer_id', 'event', 'time', 'offer_id',

        # 이벤트/거래 정보
        'amount', 'event_reward',

        # 오퍼 정보
        'offer_type', 'offer_reward', 'difficulty', 'duration', 'channels',

        # 채널 정보
        'web', 'email', 'mobile', 'social',

        # 고객 정보
        'gender', 'age', 'income', 'became_member_on',
    ]
]

In [147]:
print("transcript 원본 행 수:", len(df_transcript_clean))
print("전체 통합 행 수:", len(df_transcript_portfolio_profile))
print("행 수 차이:", len(df_transcript_portfolio_profile) - len(df_transcript_clean))

check_basic_info(df_transcript_portfolio_profile, "transcript + portfolio + profile")
check_id_duplicates(df_transcript_portfolio_profile, 'customer_id', 'transcript + portfolio + profile (customer_id)')
check_id_duplicates(df_transcript_portfolio_profile, 'offer_id', 'transcript + portfolio + profile (offer_id)')

transcript 원본 행 수: 306137
전체 통합 행 수: 306137
행 수 차이: 0

transcript + portfolio + profile의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,306137
1,열 개수,19
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
event_reward,float64,33182,10.84,272955,89.16,4
amount,float64,138953,45.39,167184,54.61,5103
offer_id,str,167184,54.61,138953,45.39,10
offer_reward,float64,167184,54.61,138953,45.39,5
difficulty,float64,167184,54.61,138953,45.39,5
duration,float64,167184,54.61,138953,45.39,5
channels,str,167184,54.61,138953,45.39,4
offer_type,str,167184,54.61,138953,45.39,3
web,float64,167184,54.61,138953,45.39,2
mobile,float64,167184,54.61,138953,45.39,2


[테이블 요약]


,customer_id,event,time,offer_id,amount,event_reward,offer_type,offer_reward,difficulty,duration,channels,web,email,mobile,social,gender,age,income,became_member_on
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,bogo,5.0,5.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,F,75.0,100000.0,2017-05-09
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,discount,5.0,20.0,10.0,"['web', 'email']",1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaT
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,discount,2.0,10.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,M,68.0,70000.0,2018-04-26
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,discount,2.0,10.0,10.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaT
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,bogo,10.0,10.0,5.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaT



transcript + portfolio + profile (customer_id)의 값 중복 확인
전체 행 수: 306137
customer_id 고유 개수: 17000
중복 customer_id 개수: 289137

transcript + portfolio + profile (offer_id)의 값 중복 확인
전체 행 수: 306137
offer_id 고유 개수: 10
중복 offer_id 개수: 306126


## profile 정보가 없는 고객 이벤트

In [148]:
profile_cols = [col for col in ['gender', 'age', 'income', 'became_member_on']
                if col in df_transcript_profile.columns]

missing_profile_mask = df_transcript_profile[profile_cols].isna().all(axis=1)
df_profile_missing_events = df_transcript_profile.loc[missing_profile_mask].copy()

missing_event_count = len(df_profile_missing_events)
total_event_count = len(df_transcript_profile)
missing_event_ratio = (missing_event_count / total_event_count) * 100

print("=" * 60)
print("profile 정보가 없는 고객 이벤트 요약")
print("=" * 60)
print(f"사용한 profile 컬럼: {profile_cols}")
print(f"이벤트 행 수: {missing_event_count}")
print(f"고객 수: {df_profile_missing_events['customer_id'].nunique()}")
print()

print("[event별 개수]")
print(df_profile_missing_events['event'].value_counts(dropna=False))
print()

print("=" * 60)
print("profile 정보가 없는 고객 이벤트 비율")
print("=" * 60)
print(f"고객 정보가 없는 이벤트 수: {missing_event_count}")
print(f"전체 이벤트 수: {total_event_count}")
print(f"비율: {missing_event_ratio:.2f}%")
print()

print("[예시 데이터]")
display(df_profile_missing_events.head())

profile 정보가 없는 고객 이벤트 요약
사용한 profile 컬럼: ['gender', 'age', 'income', 'became_member_on']
이벤트 행 수: 33749
고객 수: 2175

[event별 개수]
event
transaction        14996
offer received      9776
offer viewed        7865
offer completed     1112
Name: count, dtype: int64

profile 정보가 없는 고객 이벤트 비율
고객 정보가 없는 이벤트 수: 33749
전체 이벤트 수: 306137
비율: 11.02%

[예시 데이터]


,customer_id,event,time,offer_id,amount,event_reward,gender,age,income,became_member_on
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,NaN,NaN,NaN,NaT
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,NaN,NaN,NaN,NaT
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,NaN,NaN,NaN,NaT
6,c4863c7985cf408faee930f111475da3,offer received,0,2298d6c36e964ae4a3e7e9706d1fb8c2,NaN,NaN,NaN,NaN,NaN,NaT
10,744d603ef08c4f33af5a61c8c7628d1c,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,NaN,NaN,NaN,NaT


df_transcript_profile 기준으로 확인한 결과,\
**profile 정보가 없는 고객 이벤트는 33,749건으로 전체 이벤트의 약 11.02%**

이들을 단순 결측치로만 보기보다\
고객 정보 미기재 고객일 가능성도 있어 완전히 제거하는 것은 신중할 필요가 있다.

또한 이 이벤트에는 transaction, offer received, offer viewed, offer completed가 모두 포함되어 있어,\
이를 일괄 제거할 경우 퍼널 분석, 전환율 분석, 오퍼 반응 분석 결과가 왜곡될 수 있다.

따라서 해당 이벤트를 삭제하지 않고 유지하되,\
profile_missing = 1 또는 customer_info_group = 'Unknown'과 같은\
라벨링 컬럼을 추가하여 별도로 관리하는 방식이 적절하다.

결론
transcript 이벤트는 유지\
profile 정보가 없는 고객은 profile_missing = 1 또는 customer_info_group = 'Unknown'로 표시\
전체 행동 분석에서는 포함\
고객 특성 분석에서는 제외하거나 별도 집단으로 분리

In [149]:
profile_cols = ['gender', 'age', 'income', 'became_member_on']

# 1 = 고객 정보 있음, 0 = 고객 정보 없음
df_transcript_profile['profile_missing'] = (
    df_transcript_profile[profile_cols].notna().all(axis=1).astype('uint8')
)
# 1 = 고객 정보 있음, 0 = 고객 정보 없음
df_transcript_portfolio_profile['profile_missing'] = (
    df_transcript_portfolio_profile[profile_cols].notna().all(axis=1).astype('uint8')
)


display(df_transcript_profile.head(2))

display(df_transcript_portfolio_profile.head(2))

,customer_id,event,time,offer_id,amount,event_reward,gender,age,income,became_member_on,profile_missing
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,F,75.0,100000.0,2017-05-09,1
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,NaN,NaN,NaN,NaT,0


,customer_id,event,time,offer_id,amount,event_reward,offer_type,offer_reward,difficulty,duration,channels,web,email,mobile,social,gender,age,income,became_member_on,profile_missing
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,bogo,5.0,5.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,F,75.0,100000.0,2017-05-09,1
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,discount,5.0,20.0,10.0,"['web', 'email']",1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaT,0


## 통합 csv 파일 보고서


### 오퍼 성과 분석 → transcript_portfolio
1. 행수 증가 없음
2. customer_id 중복 존재
    - transcript는 이벤트 로그 테이블로 한 고객이 여러 번 등장하는 게 정상
    - 예시:\
    offer received/offer viewed/transaction/offer completed 를 여러 번 하면 그만큼 행이 생긴다.

#### transcript_portfolio 컬럼 해설
- df_transcript_profile
    - 역할: 고객 이벤트 로그에 고객 정보(profile)를 결합한 테이블
    - 컬럼 해설
        - customer_id : 고객을 구분하는 고유 ID
        - event : 고객 행동 또는 오퍼 반응 이벤트 유형
        - time : 데이터 내 상대 시간 정보
        - offer_id : 관련 프로모션의 고유 ID
        - amount : 거래 발생 시 구매 금액
        - event_reward : 오퍼 완료 이벤트에서 지급된 실제 보상 금액
        - gender : 고객 성별 정보
        - age : 고객 나이 정보
        - income : 고객 소득 정보
        - became_member_on : 고객 가입일 정보. 신규/기존 고객 구분, 가입 기간 계산, 코호트 분석에 활용
        - profile_missing : 고객 프로필 정보 존재 여부를 나타내는 변수 (0=정보 없음, 1=정보 있음)

### 고객 특성 분석 → transcript_profile
1. 행수 증가 없음
2. offer_id 중복 존재
    - 오퍼는 10종류 이고 그 오퍼가 여러 고객에게 여러 번 등장,\
    로그 테이블에서 offer_id가 계속 반복되는 게 정상

#### transcript_profile 컬럼 해설
- df_transcript_portfolio
    - 역할: 고객 이벤트 로그에 프로모션 정보(portfolio)를 결합한 테이블
    - 컬럼 해설
        - customer_id : 고객을 구분하는 고유 ID
        - event : 고객 행동 또는 오퍼 반응 이벤트 유형
        - time : 데이터 내 상대 시간 정보
        - offer_id : 관련 프로모션의 고유 ID
        - amount : 거래 발생 시 구매 금액
        - event_reward : 오퍼 완료 이벤트에서 지급된 실제 보상 금액
        - offer_type : 프로모션 유형
        - offer_reward : 프로모션 완료 시 제공되는 보상 금액
        - difficulty : 보상을 받기 위해 필요한 최소 지출 금액
        - duration : 프로모션 유효 기간
        - channels : 프로모션이 전달된 채널 목록
        - web : 웹 채널 포함 여부 (0=정보 없음, 1=정보 있음)
        - email : 이메일 채널 포함 여부 (0=정보 없음, 1=정보 있음)
        - mobile : 모바일 채널 포함 여부 (0=정보 없음, 1=정보 있음)
        - social : 소셜 채널 포함 여부 (0=정보 없음, 1=정보 있음)

### 세그먼트 / 종합 분석 → transcript_portfolio_profile
1. 행수 증가 없음
2. customer_id / offer_id 불일치가 이미 없었음
3. 결측 패턴도 자연스러움
    - offer_id, offer_type, offer_reward, difficulty, duration, channels\
    transaction은 원래 특정 offer와 직접 연결되지 않는 행이 많음
3. 중복 확인 결과\
한 고객은 여러 이벤트를 가진다\
같은 오퍼가 여러 고객, 여러 시점에 반복 등장함

#### transcript_portfolio_profile 컬럼 해설
- df_transcript_portfolio_profile
    - 역할: 고객 이벤트 로그에 프로모션 정보(portfolio)와 고객 정보(profile)를 함께 결합한 통합 테이블
    - 컬럼 해설
        - customer_id : 고객을 구분하는 고유 ID
        - event : 고객 행동 또는 오퍼 반응 이벤트 유형
        - time : 데이터 내 상대 시간 정보
        - offer_id : 관련 프로모션의 고유 ID
        - amount : 거래 발생 시 구매 금액
        - event_reward : 오퍼 완료 이벤트에서 지급된 실제 보상 금액
        - offer_type : 프로모션 유형
        - offer_reward : 프로모션 완료 시 제공되는 보상 금액
        - difficulty : 보상을 받기 위해 필요한 최소 지출 금액
        - duration : 프로모션 유효 기간
        - channels : 프로모션이 전달된 채널 목록
        - web : 웹 채널 포함 여부 (0=정보 없음, 1=정보 있음)
        - email : 이메일 채널 포함 여부 (0=정보 없음, 1=정보 있음)
        - mobile : 모바일 채널 포함 여부 (0=정보 없음, 1=정보 있음)
        - social : 소셜 채널 포함 여부 (0=정보 없음, 1=정보 있음)
        - gender : 고객 성별 정보
        - age : 고객 나이 정보
        - income : 고객 소득 정보
        - became_member_on : 고객 가입일 정보. 가입 기반 세그먼트, 코호트 분석, 가입 후 경과 기간 계산에 활용
        - profile_missing : 고객 프로필 정보 존재 여부를 나타내는 변수 (0=정보 없음, 1=정보 있음)

> 채널 정보(web, email, mobile, social)를 별도로 넣지 않은 이유\
web, email, mobile, social은 channels 컬럼을 0/1 형태로 나눈 파생 컬럼이므로,\
원본 의미 설명에서는 중복을 피하기 위해 별도 설명을 생략했다.

# 저장

In [150]:
# ============================================================
# 통합 파일 저장
# ============================================================
# 이벤트 + 고객 정보
df_transcript_profile.to_csv("data/transcript_profile.csv", index=False)

# 이벤트 + 오퍼 정보
df_transcript_portfolio.to_csv("data/transcript_portfolio.csv", index=False)

# 이벤트 + 오퍼 정보 + 고객 정보
df_transcript_portfolio_profile.to_csv("data/transcript_portfolio_profile.csv", index=False)

print("통합 csv 3종 저장 완료")

통합 csv 3종 저장 완료
